In [5]:
import pandas as pd
from datetime import datetime, timedelta, date
import numpy as np
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.decomposition import PCA, KernelPCA
import patsy

## Functions to ease processing

In [6]:
def preprocess(data):
    tmp = pd.to_datetime(data['Date'])
    data['Store'] = data['Store'].astype('int64')
    data['Dept'] = data['Dept'].astype('int64')
    data['Wk'] = tmp.dt.isocalendar().week
    data['Yr'] = tmp.dt.year
    data['Yr2'] = tmp.dt.year * tmp.dt.year
    data['Wk'] = pd.Categorical(data['Wk'], categories=[i for i in range(1, 53)])  # 52 weeks 
    if 'IsHoliday' in data.columns:
        data['IsHoliday'] = data['IsHoliday'].apply(int)
    return data

def clean_column_names(df):
    cols = []
    for i in df.columns:
        cols.append(i.replace("[", "").replace("]", ""))
    df.columns = cols
    #print(df.columns)
    return df

def getPCA(df):
    cols = ['Date', 'Dept'] + ([i for i in range(1, 46)])
    df_pca = pd.DataFrame(columns=cols)
    for i in df['Dept'].unique():
        # Filter rows where Dept is equal to i
        filtered_df = df[df['Dept'] == i]
        
        # Select only the columns 'Store', 'Date', and 'Weekly_Sales'
        selected_columns = filtered_df[['Store', 'Date', 'Weekly_Sales']]

        # Pivot table to spread 'Store' values into columns, with 'Weekly_Sales' as values
        df_dept_ts = selected_columns.pivot(index='Date', columns='Store', values='Weekly_Sales').reset_index()

        df_dept_arr = np.array(df_dept_ts.drop(columns='Date'))
        df_dept_arr[np.isnan(df_dept_arr)] = 0
        
        num = min(min(df_dept_ts.drop(columns='Date').shape), 8)

        pca = PCA(n_components=num, random_state=42)

        pca.fit(df_dept_arr)
        reconstructed_pca = pca.inverse_transform(pca.transform(df_dept_arr))
        df_dept_ts['Dept'] = i
        df_pca_i = pd.concat([df_dept_ts[['Date', 'Dept']], pd.DataFrame(reconstructed_pca, columns=filtered_df['Store'].unique())], axis=1)
        df_pca = pd.concat([df_pca, df_pca_i])
    df_pca = df_pca.melt(id_vars=['Date', 'Dept']).dropna()
    df_pca.columns = ['Date', 'Dept', 'Store', 'Weekly_Sales']

    return df_pca

## PCA + Linear Regression

In [7]:
num_folds = 10

for i in range(1, num_folds + 1): 
    print("Fold ", i)
    # Reading train data
    file_path = f'Proj2_Data/fold_{i}/train.csv'
    train = pd.read_csv(file_path)

    #Do pivot, PCA, then revert df back to original format 
    train = getPCA(train)
    
    # Reading test data
    file_path = f'Proj2_Data/fold_{i}/test.csv'
    test = pd.read_csv(file_path)

    # pre-allocate a pd to store the predictions
    test_pred = pd.DataFrame()

    train_pairs = train[['Store', 'Dept']].drop_duplicates(ignore_index=True)
    test_pairs = test[['Store', 'Dept']].drop_duplicates(ignore_index=True)
    unique_pairs = pd.merge(train_pairs, test_pairs, how = 'inner', on =['Store', 'Dept'])

    train_split = unique_pairs.merge(train, on=['Store', 'Dept'], how='left')
    train_split = preprocess(train_split)
    y, X = patsy.dmatrices('Weekly_Sales ~ Weekly_Sales + Store + Dept + Yr + Yr2 + Wk', 
                           data = train_split, 
                           return_type='dataframe')

    train_split = dict(tuple(X.groupby(['Store', 'Dept'])))


    test_split = unique_pairs.merge(test, on=['Store', 'Dept'], how='left')
    test_split = preprocess(test_split)
    y, X = patsy.dmatrices('Yr ~ Store + Dept + Yr + Yr2 + Wk', #'Yr ~ Store + Dept + Yr  + Wk'
                           data = test_split, 
                           return_type='dataframe')
    X['Date'] = test_split['Date']
    
    test_split = dict(tuple(X.groupby(['Store', 'Dept'])))
    
    keys = list(train_split)

    for key in keys:
        X_train = train_split[key]
        X_test = test_split[key]
        

        Y = X_train['Weekly_Sales']
        X_train = X_train.drop(['Weekly_Sales','Store', 'Dept'], axis=1)

        cols_to_drop = X_train.columns[(X_train == 0).all()]
        
        X_train = X_train.drop(columns=cols_to_drop)
        X_test = X_test.drop(columns=cols_to_drop)

        cols_to_drop = []
        for ii in range(len(X_train.columns) - 1, 1, -1):  # Start from the last column and move backward
            col_name = X_train.columns[ii]
            # Extract the current column and all previous columns
            tmp_Y = X_train.iloc[:, ii].values
            tmp_X = X_train.iloc[:, :ii].values

            coefficients, residuals, rank, s = np.linalg.lstsq(tmp_X, tmp_Y, rcond=None)
            if np.sum(residuals) < 1e-10:
                    cols_to_drop.append(col_name)

        X_train = X_train.drop(columns=cols_to_drop)
        X_test = X_test.drop(columns=cols_to_drop)

        model = sm.OLS(Y, X_train).fit()
        mycoef = model.params.fillna(0)
        
        tmp_pred = X_test[['Store', 'Dept', 'Date']]
        X_test = X_test.drop(['Store', 'Dept', 'Date'], axis=1)
        
            
        tmp_pred['Weekly_Pred'] = np.dot(X_test, mycoef)
        test_pred = pd.concat([test_pred, tmp_pred], ignore_index=True)
        if 'IsHoliday' in test_pred.columns:
            test_pred = test_pred.drop(columns="IsHoliday")
        test_pred = pd.merge(test_pred, test, on=['Store', 'Dept', 'Date'])

    test_pred['Weekly_Pred'].fillna(0, inplace=True)
    file_path = f'Proj2_Data/fold_{i}/mypred.csv'
    test_pred.to_csv(file_path, index=False)

Fold  1
Fold  2
Fold  3
Fold  4
Fold  5
Fold  6
Fold  7
Fold  8
Fold  9
Fold  10


## Post-processing for fold 5

In [8]:
def findVal(test_pred, i=4):
    file_path = 'Proj2_Data/test_with_label.csv'
    test_with_label = pd.read_csv(file_path)

    file_path = f'Proj2_Data/fold_{i+1}/test.csv'
    test = pd.read_csv(file_path)
    test = test.drop(columns=['IsHoliday']).merge(test_with_label, on=['Date', 'Store', 'Dept'])
    test['Date'] = pd.to_datetime(test['Date'])  
    

    # Left join with the test data
    test_pred['Date'] = pd.to_datetime(test_pred['Date'])  
    new_test = test_pred.merge(test, on=['Date', 'Store', 'Dept'], how='left')

    # Compute the Weighted Absolute Error
    actuals = new_test['Weekly_Sales']
    preds = new_test['Weekly_Pred']
    weights = new_test['IsHoliday_x'].apply(lambda x: 5 if x else 1)
    return (sum(weights * abs(actuals - preds)) / sum(weights))


In [10]:
test_pred = pd.read_csv(f'Proj2_Data/fold_5/mypred_copy.csv')
test_pred1 = test_pred 

num=100
r = [0.020] #[x/num for x in range(num//20)] #
vals = []


test_pred1['Date'] = pd.to_datetime(test_pred1['Date'])  
test_pred1['Wk'] = test_pred1['Date'].dt.isocalendar().week
tmp = test_pred1[test_pred1['Wk']==51].copy()
tmp['Adjustment'] = tmp['Weekly_Pred']*r
tmp = tmp[['Store', 'Dept', 'Wk', 'Adjustment']]

# Subtract from week 51
test_pred1 = pd.merge(test_pred1, tmp, on=['Store', 'Dept', 'Wk'], how='left')
test_pred1 = test_pred1.fillna(0)
test_pred1['Weekly_Pred'] = test_pred1['Weekly_Pred'] - test_pred1['Adjustment']
#print(test_pred1.head(5))
test_pred1 = test_pred1[['Store', 'Dept', 'Date', 'Weekly_Pred', 'Wk', 'IsHoliday']]



# Add to week 52
tmp['Wk'] = 52 
test_pred1 = pd.merge(test_pred1, tmp, on=['Store', 'Dept', 'Wk'], how='left')
test_pred1 = test_pred1.fillna(0)
test_pred1['Weekly_Pred'] = test_pred1['Weekly_Pred'] + test_pred1['Adjustment']
test_pred1 = test_pred1[['Store', 'Dept', 'Date', 'Weekly_Pred', 'IsHoliday']]

test = pd.read_csv(f'Proj2_Data/fold_5/test.csv')
test['Date'] = pd.to_datetime(test['Date'])  
test_pred1['Date'] = pd.to_datetime(test_pred1['Date'])  
test_pred1 = pd.merge(test_pred1.drop(columns='IsHoliday'), test, on=['Store', 'Dept', 'Date'])
test_pred1.to_csv(f'Proj2_Data/fold_5/mypred.csv', index=False)

In [11]:
def myeval():
    file_path = 'Proj2_Data/test_with_label.csv'
    test_with_label = pd.read_csv(file_path)
    num_folds = 10
    wae = []

    for i in range(num_folds):
        file_path = f'Proj2_Data/fold_{i+1}/test.csv'
        test = pd.read_csv(file_path)
        test = test.drop(columns=['IsHoliday']).merge(test_with_label, on=['Date', 'Store', 'Dept'])

        file_path = f'Proj2_Data/fold_{i+1}/mypred.csv'
        test_pred = pd.read_csv(file_path)

        # Left join with the test data
        new_test = test_pred.merge(test, on=['Date', 'Store', 'Dept'], how='left')

        # Compute the Weighted Absolute Error
        actuals = new_test['Weekly_Sales']
        preds = new_test['Weekly_Pred']
        weights = new_test['IsHoliday_x'].apply(lambda x: 5 if x else 1)
        wae.append(sum(weights * abs(actuals - preds)) / sum(weights))

    return wae

In [12]:
wae = myeval()
for value in wae:
    print(f"\t{value:.3f}")
print("Avg: ",f"{sum(wae) / len(wae):.3f}")
#2334.678 for week 5 without post
#2189.342 for week 5

	1945.900
	1364.163
	1384.109
	1528.781
	2240.364
	1638.096
	1614.847
	1355.272
	1337.454
	1334.495
Avg:  1574.348
